In [0]:
from scripts.silver import silver_loader
from scripts.utils import silver_utils

print(dir(silver_loader))
print(dir(silver_utils))

In [0]:
from pyspark.sql.functions import (
    col, when, sum, lit, trim, lower, upper,
    to_date, current_timestamp,round
)
from datetime import datetime
import uuid

In [0]:
df=silver_loader.get_incremental(spark,'users',"id")

In [0]:
silver_utils.check_schema(df)

In [0]:
df = df.drop("avatar")
df = df.drop("password")

#check changes
print(df.columns)

In [0]:
df=silver_utils.cast_types(df,{ 
    "id" : "long",
    "creationAt" : "timestamp",
    "updatedAt" : "timestamp",
    "email" : "string",
    "name" : "string",
    "role" : "string"})

In [0]:
df =silver_utils.rename_col(df,"id","user_id")
df =silver_utils.rename_col(df,"updatedAt","user_updated_at")
df =silver_utils.rename_col(df,"creationAt","user_creation_at")

In [0]:
silver_utils.null_profiling(df,"users")

df=silver_utils.handle_nulls_drop(df,drop_cols=["user_id","email"])
df= df.withColumn("is_name_null",when(col("name").isNull(),lit("yes")).otherwise(lit("no")))
df= df.withColumn("is_role_null",when(col("role").isNull(),lit("yes")).otherwise(lit("no")))

In [0]:
df = silver_utils.handle_duplicates(df,["user_id"])

In [0]:
df = silver_utils.standardize_strings(
    df,{
    "email" : lambda c : lower(trim(c)),
    "role" : lambda c : lower(trim(c)),
    "name" : lambda c : lower(trim(c))
    }
)

In [0]:
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

checks = [
    #nulls
    (
        "user_id",
        "user_id not null",
        col("user_id").isNotNull()
    ),
    (
        "email",
        "email not null",
        col("email").isNotNull()
    ),
    (
        "name",
        "name not null",
        col("name").isNotNull()
    ),
    (
        "role",
        "role not null",
        col("role").isNotNull()
    ),

    #business rules
    (
       "email","must be valid email format",col("email").rlike(email_regex)
    ),
    (
        "user_updated_at",
        "user_updated_at must be >= user_creation_at",
        col("user_updated_at") >= col("user_creation_at")
    ),
    (
        "user_creation_at","user_creation_at must be <= user_updated_at",col("user_creation_at") <= col("user_updated_at")
    )
]

dq_table = silver_utils.build_dq_table(
    spark=spark,
    df=df,
    checks=checks,
    table_name="users_api",
    warn_threshold=5.0
)

dq_table.write.mode("append").saveAsTable("workspace.ecommerce_silver.users_api_dq")
dq_saved = spark.read.table("workspace.ecommerce_silver.users_api_dq")
display(dq_saved)

In [0]:
df=df.withColumn("is_active", when(col("user_id").isNotNull() & col("email").isNotNull(), lit("yes")).otherwise(lit("no")))

In [0]:
df = silver_utils.add_silver_metadata(df)

In [0]:
#Append new rows into silver ,,first time use overwrite afteer use append
df.write.format("delta").mode("append").saveAsTable("workspace.ecommerce_silver.users")
print(" Data written to silver table successfully")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.ecommerce_silver.users LIMIT 10